# **Load data**

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/dataset/jigsaw-toxic-comment-detection-challenge-train-dataset.csv'

try:
    df = pd.read_csv(file_path)
    print("Kết nối thành công! Đã tải bộ dữ liệu Jigsaw từ Google Drive.")
    print(f"Tổng số dòng dữ liệu: {df.shape[0]:,}")
    print(f"Danh sách các cột trong file: {list(df.columns)}")

    print("\n5 dòng dữ liệu đầu tiên:")
    print(df.head())
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file tại đường dẫn: {file_path}")
    print("Vui lòng kiểm tra lại xem bạn đã upload file lên Google Drive chưa, hoặc kiểm tra xem tên file có chính xác không.")

Mounted at /content/drive
Kết nối thành công! Đã tải bộ dữ liệu Jigsaw từ Google Drive.
Tổng số dòng dữ liệu: 159,571
Danh sách các cột trong file: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

5 dòng dữ liệu đầu tiên:
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3          

# **Overall check**

## Data type, Null and Memory usage check

In [ ]:
df = pd.read_csv(file_path)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB
None


## Change `id` and `comment_text` type to string

In [ ]:
df['id'] = df['id'].astype("string")
df['comment_text'] = df['comment_text'].astype("string")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  string
 1   comment_text   159571 non-null  string
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), string(2)
memory usage: 9.7 MB
None


## Check numeric columns

In [ ]:
print(df.describe())

               toxic   severe_toxic        obscene         threat  \
count  159571.000000  159571.000000  159571.000000  159571.000000   
mean        0.095844       0.009996       0.052948       0.002996   
std         0.294379       0.099477       0.223931       0.054650   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.000000       0.000000       0.000000       0.000000   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.000000       0.000000       0.000000       0.000000   
max         1.000000       1.000000       1.000000       1.000000   

              insult  identity_hate  
count  159571.000000  159571.000000  
mean        0.049364       0.008805  
std         0.216627       0.093420  
min         0.000000       0.000000  
25%         0.000000       0.000000  
50%         0.000000       0.000000  
75%         0.000000       0.000000  
max         1.000000       1.000000  


# **EDA**

## Biểu đồ 1: Tỉ lệ toxic và non-toxic (Biểu đồ tròn)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Danh sách 6 nhãn độc hại trong bộ dữ liệu Jigsaw
target_labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Cấu hình thẩm mỹ cho tất cả các biểu đồ bằng seaborn
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.figsize"] = (10, 6)

# Định nghĩa mã màu đồng bộ để tái sử dụng
reds_palette = sns.color_palette("Reds_r", n_colors=6)
COLOR_MIXED_SEQUENTIAL = "Reds"  # Dải màu đồng bộ từ nhạt đến đậm cho heatmap/bar

# Một bình luận được coi là toxic nếu nó dính ít nhất 1 trong 6 nhãn
df["is_toxic"] = df[target_labels].any(axis=1).astype(int)
toxic_counts = df["is_toxic"].value_counts()

fig, ax = plt.subplots()
ax.pie(
    toxic_counts,
    labels=["Lành mạnh (Non-toxic)", "Độc hại (Toxic)"],
    autopct="%1.1f%%",
    startangle=90,
    colors=[reds_palette[5], reds_palette[0]],
    explode=(0, 0.1),
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)
ax.set_title("Tỷ Lệ Bình Luận Độc Hại vs Lành Mạnh Trong Tập Dữ Liệu", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig("5_1_toxic_vs_nontoxic.png", dpi=300)
plt.close()

## Biểu đồ 2: Phân bố 6 Nhãn tiêu cực (Biểu đồ cột)

In [ ]:
label_counts = df[target_labels].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
sns.barplot(
    x=label_counts.index,
    y=label_counts.values,
    ax=ax,
    palette=sns.color_palette("Reds_r", n_colors=len(target_labels)),
)

# Hiển thị số liệu cụ thể trên đầu mỗi cột
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="center",
        xytext=(0, 8),
        textcoords="offset points",
        fontsize=10,
    )

ax.set_title("Tần Suất Xuất Hiện Của Sáu Nhãn Độc Hại", fontsize=14, pad=20)
ax.set_xlabel("Các Loại Nhãn Tiêu Cực", fontsize=12)
ax.set_ylabel("Số Lượng Bình Luận", fontsize=12)
ax.set_ylim(0, label_counts.max() * 1.15)  # Tạo khoảng trống phía trên để ghi số liệu
plt.tight_layout()
plt.savefig("5_2_label_distribution.png", dpi=300)
plt.close()

/tmp/ipykernel_2483/2458965294.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


## Biểu đồ 3: Độ dài bình luận (Biểu đồ phân phối mật độ - Histogram)

In [ ]:
df["word_count"] = df["comment_text"].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots()
# Giới hạn độ dài ở mức 400 từ để biểu đồ trực quan, tránh các bài viết dài đột biến làm loãng đồ thị
sns.histplot(
    data=df[df["word_count"] <= 400],
    x="word_count",
    kde=True,
    color=reds_palette[0],  # Màu đường KDE (Đỏ sẫm)
    fill=True,
    facecolor=reds_palette[5],  # Màu nền cột (Đỏ cực nhạt sát trắng)
    bins=50,
    ax=ax,
)

ax.set_title("Phân Phối Độ Dài Bình Luận (Giới hạn dưới 400 từ)", fontsize=14, pad=20)
ax.set_xlabel("Số Lượng Từ Trong Một Bình Luận", fontsize=12)
ax.set_ylabel("Số Lượng Mẫu Dữ Liệu", fontsize=12)
plt.tight_layout()
plt.savefig("5_3_comment_length_distribution.png", dpi=300)
plt.close()

print(df["word_count"].describe())

count    159571.000000
mean         67.273527
std          99.230702
min           1.000000
25%          17.000000
50%          36.000000
75%          75.000000
max        1411.000000
Name: word_count, dtype: float64


## Biểu đồ 4: Mối quan hệ giữa các nhãn (Biểu đồ ma trận nhiệt - Heatmap)

In [ ]:
# Tính toán ma trận hệ số tương quan giữa 6 nhãn
correlation_matrix = df[target_labels].corr()

fig, ax = plt.subplots(figsize=(8, 6))
# Sử dụng mặt nạ (mask) để ẩn một nửa ma trận đối xứng, giúp biểu đồ thoáng hơn
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap=COLOR_MIXED_SEQUENTIAL,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax,
)

ax.set_title("Ma Trận Tương Quan Hệ Số Giữa Sáu Nhãn Độc Hại", fontsize=14, pad=20)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("5_4_labels_correlation_heatmap.png", dpi=300)
plt.close()

## Nhãn `toxic` luôn bằng 1 khi bình luận đó là độc hại?  

In [ ]:
five_sub_labels = ["severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Bước 1: Lọc ra các dòng mà ít nhất 1 trong 5 nhãn cuối bằng 1
has_sub_toxicity = df[five_sub_labels].any(axis=1)
df_sub = df[has_sub_toxicity]

# Bước 2: Đếm xem trong tập con này, có bao nhiêu dòng có nhãn 'toxic' bằng 1 và bằng 0
toxic_distribution = df_sub['toxic'].value_counts()

print("--- KẾT QUẢ KIỂM CHỨNG GIẢ THUYẾT ---")
print(f"Tổng số mẫu có ít nhất 1 trong 5 nhãn cuối: {len(df_sub):,}")
print(f"Số mẫu có nhãn toxic = 1: {toxic_distribution.get(1, 0):,}")
print(f"Số mẫu có nhãn toxic = 0 (Ngoại lệ): {toxic_distribution.get(0, 0):,}")

--- KẾT QUẢ KIỂM CHỨNG GIẢ THUYẾT ---
Tổng số mẫu có ít nhất 1 trong 5 nhãn cuối: 10,559
Số mẫu có nhãn toxic = 1: 9,628
Số mẫu có nhãn toxic = 0 (Ngoại lệ): 931


## Số lượng bình luận độc hại chỉ có nhãn `toxic`

In [ ]:
five_sub_labels = ["severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Bước 1: Tìm các dòng có toxic = 1
is_toxic = df['toxic'] == 1

# Bước 2: Tìm các dòng mà CẢ 5 nhãn còn lại ĐỀU BẰNG 0
no_sub_toxicity = df[five_sub_labels].sum(axis=1) == 0

# Bước 3: Lọc ra tập dữ liệu thỏa mãn cả 2 điều kiện (Chỉ có duy nhất nhãn toxic)
only_toxic_df = df[is_toxic & no_sub_toxicity]

# Tính toán số liệu thống kê
total_toxic = is_toxic.sum()
only_toxic_count = len(only_toxic_df)
percentage = (only_toxic_count / total_toxic) * 100

print("--- KẾT QUẢ KIỂM CHỨNG TÌNH HUỐNG CHỈ CÓ NHÃN TOXIC ---")
print(f"Tổng số bình luận có nhãn toxic = 1: {total_toxic:,}")
print(f"Số lượng bình luận CHỈ có duy nhất nhãn toxic: {only_toxic_count:,}")
print(f"Tỷ lệ trường hợp chỉ dính nhãn toxic trên tổng số toxic: {percentage:.2f}%")

--- KẾT QUẢ KIỂM CHỨNG TÌNH HUỐNG CHỈ CÓ NHÃN TOXIC ---
Tổng số bình luận có nhãn toxic = 1: 15,294
Số lượng bình luận CHỈ có duy nhất nhãn toxic: 5,666
Tỷ lệ trường hợp chỉ dính nhãn toxic trên tổng số toxic: 37.05%


# Loại bỏ các thông tin không cần thiết

In [ ]:
import re

# Định nghĩa các mẫu Regex cho từng đối tượng cần tìm
patterns = {
    'URL': r'https?://\S+|www\.\S+',
    'IP_Address': r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
    'Email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',  # Email
    'HTML_Tag': r'<[^>]+>',

    # Mở rộng Regex để bắt tất cả các liên kết không gian tên (Namespace) của Wikipedia
    'Wiki_Link': r'\b(?:User|Wikipedia|Help|File|Talk):[A-Za-z0-9_-]+',

    # Sử dụng Lookbehind (?<=\s|^) thay vì \b ở phía trước để tránh sót lỗi
    # Thư viện re không cho phép sử dụng Lookbehind với độ dài thay đổi nên ta thay bằng non-capturing group (?:^|(?<=\s))
    'Mention': r'(?:^|(?<=\s))@[A-Za-z0-9_-]+\b',
    'Hashtag': r'(?:^|(?<=\s))#[A-Za-z0-9_-]+\b',

    'Escape_Sequence': r'[\n\t\r]|\\n|\\t|\\r'    # Ký tự xuống dòng, tab ẩn hoặc hiển thị dạng text
}

print("=== BÁO CÁO EDA: KIỂM TRA VÀ CHỨNG MINH SỰ TỒN TẠI ===")

# Quét tập dữ liệu và thống kê số lượng
for name, pattern in patterns.items():
    # Kiểm tra dòng nào khớp với regex
    matches = df['comment_text'].str.contains(pattern, regex=True, flags=re.IGNORECASE)
    count = matches.sum()

    print(f"\n[+] Tìm thấy {count} bình luận có chứa [{name}]")

    # Nếu tìm thấy, in ra ví dụ trực quan (Sử dụng repr() để chứng minh ký tự ẩn như escape)
    if count > 0:
        sample_text = df[matches]['comment_text'].iloc[0]
        print(f"    - Ví dụ chuỗi gốc: {sample_text}")
        print(f"    - Chứng minh (repr): {repr(sample_text)}")

=== BÁO CÁO EDA: KIỂM TRA VÀ CHỨNG MINH SỰ TỒN TẠI ===

[+] Tìm thấy 5114 bình luận có chứa [URL]
    - Ví dụ chuỗi gốc: "

 Snowflakes are NOT always symmetrical! 

Under Geometry it is stated that ""A snowflake always has six symmetric arms."" This assertion is simply not true! According to Kenneth Libbrecht, ""The rather unattractive irregular crystals are by far the most common variety."" http://www.its.caltech.edu/~atomic/snowcrystals/myths/myths.htm#perfection Someone really need to take a look at his site and get FACTS off of it because I still see a decent number of falsities on this page. (forgive me Im new at this and dont want to edit anything)"
    - Chứng minh (repr): '"\n\n Snowflakes are NOT always symmetrical! \n\nUnder Geometry it is stated that ""A snowflake always has six symmetric arms."" This assertion is simply not true! According to Kenneth Libbrecht, ""The rather unattractive irregular crystals are by far the most common variety."" http://www.its.caltech.edu/~at

## Vẽ biểu đồ thống kê

In [ ]:
stats = {}
for name, pattern in patterns.items():
    # Kiểm tra dòng nào khớp với regex, trả về chuỗi Boolean True/False
    matches = df["comment_text"].str.contains(pattern, regex=True, flags=re.IGNORECASE)
    stats[name] = matches.sum()

# Sắp xếp giảm dần để biểu đồ đẹp hơn
sorted_stats = pd.Series(stats).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    x=sorted_stats.index,
    y=sorted_stats.values,
    ax=ax,
    palette=sns.color_palette("Reds_r", n_colors=len(sorted_stats)),
)

for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="center",
        xytext=(0, 8),
        textcoords="offset points",
        fontsize=10,
        weight="bold",
    )

ax.set_title(
    "Tần Suất Xuất Hiện Của Dữ Liệu Nhiễu Trong Tập Dữ Liệu Jigsaw",
    fontsize=14,
    pad=20,
    weight="bold",
)
ax.set_xlabel("Các Loại Dữ Liệu Nhiễu Cần Xử Lý", fontsize=12)
ax.set_ylabel("Số Lượng Bình Luận Chứa Nhiễu", fontsize=12)
ax.set_xticklabels(sorted_stats.index, rotation=30, ha="right")
ax.set_ylim(0, sorted_stats.max() * 1.15)  # Tạo khoảng trống phía trên
plt.tight_layout()
plt.savefig("5_5_noise_information_distribution.png", dpi=300)
plt.close()

/tmp/ipykernel_2483/322875640.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_2483/322875640.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sorted_stats.index, rotation=30, ha="right")


# **Cleaning**

In [ ]:
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, auc, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import nltk

# Download the 'wordnet' resource for NLTK
nltk.download('wordnet')

# Khởi tạo TweetTokenizer với tính năng rút gọn dấu câu lặp lại (ví dụ: !!!!! -> !!!)
# Đặt preserve_case=False để tự động chuyển về chữ thường (Lower case) khi tokenize
tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True)

# Khởi tạo WordNetLemmatizer với tính năng tìm từ gốc (ví dụ 'studies'->'study')
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Thay thế thực thể bằng Token đặc biệt để giữ ngữ cảnh câu
    text = re.sub(patterns['URL'], ' [URL] ', text)
    text = re.sub(patterns['IP_Address'], ' [IP] ', text)
    text = re.sub(patterns['Email'], " [EMAIL] ", text)
    text = re.sub(patterns['Wiki_Link'], " [WIKI_LINK] ", text)
    text = re.sub(patterns['Mention'], ' [USER] ', text)
    text = re.sub(patterns['Hashtag'], ' [HASHTAG] ', text)

    # Xóa hoàn toàn thẻ HTML và ký tự Escape
    text = re.sub(patterns['HTML_Tag'], ' ', text)
    text = re.sub(patterns['Escape_Sequence'], ' ', text)

    # Xóa các con số thuần túy
    text = re.sub(r'\b\d+ \b', ' ', text)

    # Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def custom_tokenizer(text):
    # Bước này nhận vào văn bản đã qua hàm clean_text và tiến hành tách từ
    tokens = tokenizer.tokenize(text)

    # Lemmatize từng từ ngay lập tức bằng List Comprehension
    # Loại bỏ các token có độ dài bằng 1 (như dấu câu đơn lẻ, ký tự thừa) để tiết kiệm thêm feature
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens if len(token) > 1 or token in ['!', '?']]

    return lemmatized_tokens

# Áp dụng hàm clean
df['cleaned_text'] = df['comment_text'].map(clean_text)

# Split Train / Validation (80/20) để tìm ngưỡng tối ưu trên tập Val
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Cấu hình TF-IDF
word_vectorizer = TfidfVectorizer(
    tokenizer=custom_tokenizer,
    min_df=5,            # Đổi thành 3 với tập dữ liệu Jigsaw thật của bạn
    max_df=0.9,          # Loại bỏ các từ xuất hiện > 90% số câu
    ngram_range=(1, 2),  # Giữ lại cụm 2 từ để không bị mất ý nghĩa từ phủ định (ví dụ: "not idiot")
    sublinear_tf=True,   # Áp dụng scale logarit cho tần suất từ (rất tốt cho văn bản dài)
)

X_train = word_vectorizer.fit_transform(train_df['comment_text'])
X_val = word_vectorizer.transform(val_df['comment_text'])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# **Training**

In [ ]:
# Định nghĩa Hàm Huấn luyện Naive Bayes + Logistic Regression (NBSVM)
# Kỹ thuật này nhân ma trận TF-IDF với tỷ lệ log-count của Naive Bayes trước khi đưa vào LR
def get_nb_ratio(x, y):
    # Tính xác suất Naive Bayes cơ bản
    p = x[y == 1].sum(axis=0) + 1
    q = x[y == 0].sum(axis=0) + 1
    p = p / np.sum(p)
    q = q / np.sum(q)
    return np.log(p / q)

# Lập trình vòng lặp huấn luyện và tìm ngưỡng tối ưu cho 6 nhãn
best_thresholds = {}
val_predictions_proba = {}

print("--- Bắt đầu huấn luyện NBSVM cho 6 nhãn ---")
for cls in target_labels:
    y_train_cls = train_df[cls].values
    y_val_cls = val_df[cls].values

    # BƯỚC NAIVE BAYES: Tính toán toán tử NB ratio cho nhãn hiện tại
    r = get_nb_ratio(X_train, y_train_cls)

    # Biến đổi ma trận đặc trưng bằng cách nhân chập với trọng số Naive Bayes
    X_train_nb = X_train.multiply(r)
    X_val_nb = X_val.multiply(r)

    # BƯỚC LOGISTIC REGRESSION: Huấn luyện trên ma trận đã biến đổi
    # Thuật toán 'liblinear' rất mạnh và ổn định cho ma trận sau nhân chập NB
    model = LogisticRegression(C=4.0, dual=False, solver='liblinear', max_iter=200, random_state=42)
    model.fit(X_train_nb, y_train_cls)

    # Dự đoán xác suất (%) trên tập Validation
    preds_proba = model.predict_proba(X_val_nb)[:, 1]
    val_predictions_proba[cls] = preds_proba

    # SỬ DỤNG PR-AUC ĐỂ TÌM NGƯỠNG (THRESHOLD) TỐT NHẤT CHO F1-SCORE
    # Đường cong Precision-Recall chuẩn hơn ROC-AUC rất nhiều khi dữ liệu lệch nhãn (Imbalanced data)
    precisions, recalls, thresholds = precision_recall_curve(y_val_cls, preds_proba)

    # Tính giá trị PR-AUC (Diện tích dưới đường cong Precision-Recall)
    # Lưu ý: Hàm auc nhận vào trục X trước (recalls) và trục Y sau (precisions) [1]
    pr_auc_score = auc(recalls, precisions)

    # Tính F1-score tương ứng với từng ngưỡng cắt thu được từ đồ thị PR
    # Tránh chia cho 0 nếu cả precision và recall đều bằng 0
    f1_scores = np.divide(
        2 * (precisions * recalls),
        (precisions + recalls),
        out=np.zeros_like(precisions),
        where=(precisions + recalls) != 0
    )

    # Tìm vị trí có F1-score cao nhất (bỏ phần tử cuối cùng của precisions/recalls vì không có threshold tương ứng)
    best_idx = np.argmax(f1_scores[:-1])
    best_thresh = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    # Lưu lại cấu hình tối ưu
    best_thresholds[cls] = best_thresh

    # Đánh giá lại kết quả tại chính Ngưỡng (Threshold) được chọn
    # Vì hàm precision_recall_curve dùng '>' để so sánh với ngưỡng nhằm tối ưu tốc độ tính toán
    # mà trong thực tế thì thường dùng '>=' để so sánh với ngưỡng hơn nên ta cần tính lại
    final_preds = (preds_proba >= best_thresh).astype(int)
    final_precision = precision_score(y_val_cls, final_preds, zero_division=0)
    final_recall = recall_score(y_val_cls, final_preds, zero_division=0)
    final_f1 = f1_score(y_val_cls, final_preds, zero_division=0)

    # In báo cáo kết quả chi tiết từng nhãn
    print(f"\n==========================================")
    print(f"NHÃN: [{cls.upper()}]")
    print(f"------------------------------------------")
    print(f"  • Diện tích PR-AUC:          {pr_auc_score:.4f}")
    print(f"  • Ngưỡng cắt (Threshold):    {best_thresh:.4f}")
    print(f"  • Chỉ số tại ngưỡng được chọn:")
    print(f"    - F1-Score:              {final_f1:.4f}")
    print(f"    - Precision (Độ chính xác): {final_precision:.4f}")
    print(f"    - Recall (Độ bao phủ):      {final_recall:.4f}")

print(f"\n==========================================")
print("--- Toàn bộ quá trình hoàn thành ---")

print("\n--- Pipeline Hoàn Thành ---")
print("Bảng tra cứu ngưỡng cắt tối ưu cho hệ thống thực tế (Inference):")
print(best_thresholds)

--- Bắt đầu huấn luyện NBSVM cho 6 nhãn ---

NHÃN: [TOXIC]
------------------------------------------
  • Diện tích PR-AUC:          0.8828
  • Ngưỡng cắt (Threshold):    0.2784
  • Chỉ số tại ngưỡng được chọn:
    - F1-Score:              0.8075
    - Precision (Độ chính xác): 0.8313
    - Recall (Độ bao phủ):      0.7850

NHÃN: [SEVERE_TOXIC]
------------------------------------------
  • Diện tích PR-AUC:          0.4076
  • Ngưỡng cắt (Threshold):    0.1717
  • Chỉ số tại ngưỡng được chọn:
    - F1-Score:              0.4807
    - Precision (Độ chính xác): 0.4444
    - Recall (Độ bao phủ):      0.5234

NHÃN: [OBSCENE]
------------------------------------------
  • Diện tích PR-AUC:          0.8878
  • Ngưỡng cắt (Threshold):    0.2205
  • Chỉ số tại ngưỡng được chọn:
    - F1-Score:              0.8240
    - Precision (Độ chính xác): 0.8405
    - Recall (Độ bao phủ):      0.8082

NHÃN: [THREAT]
------------------------------------------
  • Diện tích PR-AUC:          0.4601
  • Ngư

# **Save model**

In [ ]:
import joblib

# Tạo một cấu trúc Dictionary để đóng gói tất cả những gì cần thiết
artifacts = {
    "vectorizer": word_vectorizer,
    "list_classes": target_labels,
    "nb_ratios": {},       # Lưu trọng số Naive Bayes của từng nhãn
    "models": {},          # Lưu mô hình Logistic Regression của từng nhãn
    "thresholds": {}       # Lưu ngưỡng cắt tối ưu của từng nhãn
}

for cls in target_labels:
    # Giả sử 'r' là trọng số NB, 'model' là LR, 'best_thresh' là ngưỡng bạn đã tìm được ở vòng lặp trước
    artifacts["nb_ratios"][cls] = r
    artifacts["models"][cls] = model
    artifacts["thresholds"][cls] = best_thresh

# Lưu toàn bộ vào một file duy nhất với cơ chế nén để tiết kiệm dung lượng ổ cứng
joblib.dump(artifacts, "jigsaw_nbsvm_pipeline.pkl", compress=3)
print("--- Đã đóng gói và lưu mô hình thành công vào file 'jigsaw_nbsvm_pipeline.pkl' ---")


--- Đã đóng gói và lưu mô hình thành công vào file 'jigsaw_nbsvm_pipeline.pkl' ---


# **Deploy**

In [ ]:
import joblib
import pandas as pd
import numpy as np

def deploy_inference(test_csv_path, output_csv_path):
    print("--- 1. Đang nạp mô hình đã đóng gói... ---")
    # Tải file artifacts lên RAM, mất chưa tới 1 giây
    artifacts = joblib.load("jigsaw_nbsvm_pipeline.pkl")

    word_vectorizer = artifacts["vectorizer"]
    list_classes = artifacts["list_classes"]

    print("--- 2. Đang đọc tập dữ liệu Test... ---")
    # Giả sử tập test của bạn có cột 'comment_text' và cột 'id'
    test_df = pd.read_csv(test_csv_path)

    # Bước Vector hóa: Chỉ dùng hàm .transform(), TUYỆT ĐỐI không dùng .fit_transform()
    X_test = word_vectorizer.transform(test_df['comment_text'])

    # Tạo DataFrame mới để lưu kết quả dự đoán cuối cùng (0 hoặc 1)
    submission_df = pd.DataFrame()
    submission_df['id'] = test_df['id'] if 'id' in test_df.columns else test_df.index

    print("--- 3. Đang tiến hành dự đoán và áp ngưỡng cho từng nhãn... ---")
    for cls in list_classes:
        # Lấy ra các linh kiện tương ứng của nhãn đó
        r = artifacts["nb_ratios"][cls]
        model = artifacts["models"][cls]
        thresh = artifacts["thresholds"][cls]

        # Biến đổi ma trận test theo trọng số Naive Bayes của nhãn tương ứng
        X_test_nb = X_test.multiply(r)

        # Dự đoán xác suất (%)
        preds_proba = model.predict_proba(X_test_nb)[:, 1]

        # ÁP DỤNG NGƯỠNG CẮT TỐI ƯU: Nếu vượt ngưỡng thì là 1 (Độc hại), ngược lại là 0
        #submission_df[cls] = (preds_proba >= thresh).astype(int)

    # Xuất ra file CSV kết quả
    submission_df.to_csv(output_csv_path, index=False)
    print(f"--- Hoàn thành! Kết quả đã được lưu tại: {output_csv_path} ---")

if __name__ == "__main__":
    # Đường dẫn file dữ liệu thật của bạn
    deploy_inference(test_csv_path="test.csv", output_csv_path="submission_final.csv")


--- 1. Đang nạp mô hình đã đóng gói... ---
--- 2. Đang đọc tập dữ liệu Test... ---
--- 3. Đang tiến hành dự đoán và áp ngưỡng cho từng nhãn... ---
--- Hoàn thành! Kết quả đã được lưu tại: submission_final.csv ---
